In [33]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, explode, lower, regexp_replace, split
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import os

# Set SPARK_MASTER_URL from environment variable
spark_master_url = os.environ.get("SPARK_MASTER_URL", "local[*]")

spark = SparkSession.builder \
    .appName("MyPySparkApp") \
    .master(spark_master_url) \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Scala version: {spark.sparkContext.getConf().get('spark.scala.version')}")

Spark version: 3.5.0
Scala version: None


In [13]:
# --- Configuration ---
KAFKA_BOOTSTRAP_SERVERS = "kafka:9092" # Docker Compose service name and port
KAFKA_INPUT_TOPIC = "api_events"
KAFKA_OUTPUT_TOPIC = "word_counts_output"

In [ ]:
# Set up SparkSession
# The appName is useful for identifying your application in the Spark UI
# .config() is where you'd add any specific Spark configurations if needed
# spark = SparkSession.builder \
#     .appName("GNewsWordCountStreaming") \
#     .getOrCreate()

In [14]:
# Set Spark logging level to WARN to reduce verbosity
spark.sparkContext.setLogLevel("WARN")

logger = spark.sparkContext._jvm.org.apache.log4j.LogManager.getLogger(__name__)
logger.warn("SparkSession and Logger initialized.")
logger.warn(f"Reading from Kafka topic: {KAFKA_INPUT_TOPIC}")
logger.warn(f"Writing to Kafka topic: {KAFKA_OUTPUT_TOPIC}")

In [15]:
news_schema = StructType([
    StructField("title", StringType(), True),
    StructField("description", StringType(), True),
    StructField("url", StringType(), True),
    StructField("publishedAt", StringType(), True) # Keeping as StringType for now, can convert to TimestampType if needed
])

In [16]:
# --- Read from Kafka ---
# `spark.readStream` creates a DataFrame representing the unbounded table of stream data
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_INPUT_TOPIC) \
    .option("startingOffsets", "earliest").load()

logger.warn("Kafka stream loaded. Processing data...")

In [19]:
# --- Process the Kafka messages ---
# 1. Cast the 'value' column (which is binary) to a String
# 2. Parse the JSON string using the defined schema
parsed_df = kafka_df.selectExpr("CAST(value AS STRING) as json_value") \
    .select(from_json(col("json_value"), news_schema).alias("news_article"))

# 3. Extract text content (e.g., from 'title' and 'description')
#    Combine title and description for word count, handle potential nulls
text_df = parsed_df.select(
    col("news_article.title"),
    col("news_article.description")
)

In [24]:
text_df_query = text_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .trigger(processingTime="5 seconds") \
    .option("truncate", "false").start()

In [34]:
# 4. Clean and tokenise the text for word count
#    - Coalesce title and description to handle cases where one might be null
#    - Convert to lowercase
#    - Remove punctuation and split into words
words_df = text_df.select(
    explode(
        # Combine title and description, handle None by using empty string
        regexp_replace(
            lower(
                col("title")
                .cast(StringType()) # Ensure it's string, handles potential non-string types
                .cast("string") # Ensure it's string
                + " " +
                col("description")
                .cast(StringType()) # Ensure it's string, handles potential non-string types
                .cast("string") # Ensure it's string
            ),
            "[^a-z\\s]", # Regex: anything not a lowercase letter or space
            ""
        ).split("\\s+") # Split by one or more spaces
    ).alias("word")
).where(col("word") != "") # Filter out empty strings that might result from splitting

TypeError: 'Column' object is not callable

In [ ]:
# 5. Perform word count
word_counts = words_df.groupBy("word").count()

logger.warn("Word count logic defined.")


# DRAFT

In [31]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, explode, lower, regexp_replace, coalesce, lit, split
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# --- Configuration ---
KAFKA_BOOTSTRAP_SERVERS = "kafka:9092" # Docker Compose service name and port
KAFKA_INPUT_TOPIC = "api_events"
KAFKA_OUTPUT_TOPIC = "word_counts_output"
# IMPORTANT: This checkpoint location is crucial for streaming applications.
# If you want to reprocess data from the beginning of Kafka, DELETE the contents
# of this directory before restarting your Spark job.
CHECKPOINT_LOCATION = "/tmp/spark-kafka-wordcount-checkpoint"

# Set up SparkSession
spark = SparkSession.builder \
    .appName("GNewsWordCountStreaming") \
    .getOrCreate()

# Set Spark logging level to WARN to reduce verbosity
spark.sparkContext.setLogLevel("WARN")

logger = spark.sparkContext._jvm.org.apache.log4j.LogManager.getLogger(__name__)
logger.warn("SparkSession and Logger initialized.")
logger.warn(f"Reading from Kafka topic: {KAFKA_INPUT_TOPIC}")
logger.warn(f"Writing to Kafka topic: {KAFKA_OUTPUT_TOPIC}")
logger.warn(f"Using checkpoint location: {CHECKPOINT_LOCATION}")


# --- Define Schema for incoming Kafka JSON messages ---
news_schema = StructType([
    StructField("title", StringType(), True),
    StructField("description", StringType(), True),
    StructField("url", StringType(), True),
    StructField("publishedAt", StringType(), True)
])

# --- Read from Kafka ---
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_INPUT_TOPIC) \
    .option("startingOffsets", "earliest").load()

logger.warn("Kafka stream loaded. Processing data...")

# --- Process the Kafka messages ---
# 1. Cast the 'value' column (which is binary) to a String
# 2. Parse the JSON string using the defined schema
parsed_df = kafka_df.selectExpr("CAST(value AS STRING) as json_value") \
    .select(from_json(col("json_value"), news_schema).alias("news_article"))

# 3. Extract text content (e.g., from 'title' and 'description')
#    Combine title and description for word count, handle potential nulls
text_df = parsed_df.select(
    col("news_article.title"),
    col("news_article.description")
)

# 4. Clean and tokenise the text for word count
#    - Use coalesce to handle potential nulls in title/description by replacing with empty string
#    - Convert to lowercase
#    - Remove punctuation and split into words
words_df = text_df.select(
    explode(
        # The result of regexp_replace is a Column.
        # You need to apply the split function from pyspark.sql.functions to that Column.
        split( # <--- CORRECTED LINE: Call split as a function
            regexp_replace(
                lower(
                    coalesce(col("title"), lit(""))
                    + " " +
                    coalesce(col("description"), lit(""))
                ),
                "[^a-z\\s]", # Regex: anything not a lowercase letter or space
                ""
            ),
            "\\s+" # The delimiter for split
        )
    ).alias("word")
).where(col("word") != "") 

# 5. Perform word count
word_counts = words_df.groupBy("word").count()

logger.warn("Word count logic defined.")

# ---

# ## Outputting Streaming Data for Observation

# Here's how you can view the output of your streaming DataFrames.

# ### 1. Viewing `text_df` (Raw Article Content)

# This stream will show the `title` and `description` of each news article as it's processed. It uses `append` mode because it's simply adding new rows.

# ```python
# text_df_query = text_df.writeStream \
#     .outputMode("append") \
#     .format("console") \
#     .trigger(processingTime="5 seconds") # Process every 5 seconds
#     .option("truncate", "false") # Show full content of title/description
#     .start()

# logger.warn("text_df stream started for console output.")

# This is the correct way to print the word counts to your console
word_counts_query = word_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .trigger(processingTime="5 seconds") \
    .option("checkpointLocation", "/tmp/spark-kafka-wordcount-checkpoint/word_counts_console_sink") \
    .start()

# Keep the application running so the stream continues to process and print
word_counts_query.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 